Import Necessary Libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import shutil
import arcpy
import os

Point arcpy to folder containing copy of TAZ and Microzones which is the workspace

In [2]:
arcpy.env.workspace = r"D:\ABM-INP-Microzones\Scripts\Scratch\Microzone_Cleanup\Microzone_Cleanup.gdb"

**Run Identify Tool**

This will fill in all gaps between polygons

In [ ]:
arcpy.analysis.Identity(in_features="TAZ_Export", identity_features="microzones_Export", out_feature_class="Microzones_Identity")

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzone_Cleanup\\Microzone_Cleanup.gdb\\Microzones_Identity'>

**Multipart to Singlepart** 

split all of the new features into their own polygon

In [7]:
arcpy.management.MultipartToSinglepart(in_features="Microzones_Identity", out_feature_class="micro_identity_singlepart")

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzone_Cleanup\\Microzone_Cleanup.gdb\\micro_identity_singlepart'>

## Create new feild for filled gap polygon

**For some reason this was selecting MAG areas too so I just did this in ArcGIS GUI**

In [ ]:
#sql_query = "MAZID = 0 And PLANAREA = 'WFRC'"
#arcpy.management.SelectLayerByAttribute(in_layer_or_view="micro_identity_singlepart", selection_type="NEW_SELECTION", where_clause=sql_query)
#arcpy.management.CalculateField(in_table="micro_identity_singlepart", field="gap_flag", expression="1", expression_type="PYTHON3")


<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzone_Cleanup\\Microzone_Cleanup.gdb\\micro_identity_singlepart'>

## Identify zones for review

gap_flag = 1 (text column)

or 

PolsbyPopper is greater than 0 and less than .03

or 

PerimtertoArea is greater than 0 and less than .011



In [ ]:
fc = "micro_identity_singlepart"

# arcpy.management.AddField(
#     in_table=fc, 
#     field_name="Sliver_Flag", 
#     field_type="SHORT"
# )

arcpy.management.SelectLayerByAttribute(
    in_layer_or_view=fc, 
    selection_type="CLEAR_SELECTION"
)

sql_query = gap_flag = "'1' OR (PolsbyPopper > 0 AND PolsbyPopper < 0.03) OR (PerimtertoArea > 0 AND PerimtertoArea < 0.011)"


arcpy.management.SelectLayerByAttribute(
    in_layer_or_view=fc, 
    selection_type="NEW_SELECTION", 
    where_clause=sql_query
)

arcpy.management.CalculateField(
    in_table=fc, 
    field="Sliver_Flag", 
    expression="1", 
    expression_type="PYTHON3"
)


<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzone_Cleanup\\Microzone_Cleanup.gdb\\micro_identity_singlepart'>

**Above code was not respecting the selecty layer by attribute and just making everything 1 so I used arcgis gui**

## Split corridor polygons

Creates a new layer just for the sliver zones

In [ ]:
#creates a scratch layer to seperate the identifed slivers and good zones
arcpy.env.overwriteOutput = True

layer_name = "working_layer"
arcpy.management.MakeFeatureLayer(fc, layer_name)

arcpy.management.SelectLayerByAttribute(layer_name, "NEW_SELECTION", "Sliver_Flag = 1")

scratch_corridors = "scratch_corridors"

arcpy.management.CopyFeatures(layer_name, scratch_corridors)

<Result 'in_memory\\good_zones'>

Create a new layer just for the good zones

In [40]:
layer_name_2 = "working_layer_good"
arcpy.management.MakeFeatureLayer(fc, layer_name_2)

arcpy.management.SelectLayerByAttribute(layer_name_2, "NEW_SELECTION", "Sliver_Flag = 0 OR Sliver_Flag IS NULL")

arcpy.management.CopyFeatures(layer_name_2, "good_zones")

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzone_Cleanup\\Microzone_Cleanup.gdb\\good_zones'>

In [26]:
layer_name = "working_layer2" 

arcpy.management.MakeFeatureLayer(fc, layer_name)

arcpy.management.SelectLayerByAttribute(layer_name, "NEW_SELECTION", '"Sliver_Flag" = 1')
arcpy.management.CopyFeatures(layer_name, "scratch_corridors")


<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzone_Cleanup\\Microzone_Cleanup.gdb\\scratch_corridors'>

Recombines the good zones and the new subdivided slivers into a new fc

In [27]:
final_merged_fc = "microzones_ready_for_eliminate3"
arcpy.management.Merge(
    inputs=[good_zones, "scratch_corridors"], 
    output=final_merged_fc
)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzone_Cleanup\\Microzone_Cleanup.gdb\\microzones_ready_for_eliminate3'>

## Run Eliminate

Create Taz vector lines

In [51]:
taz_lines = "taz_borders"
arcpy.management.PolygonToLine(taz_fc, taz_lines)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzone_Cleanup\\Microzone_Cleanup.gdb\\taz_borders'>

Run Innital Eliminate 

This leaves us still with 174 left over sliver zones due to chaining and also micro overlap with taz lines

In [ ]:
arcpy.management.MakeFeatureLayer(fc, "micro_layer")

arcpy.management.SelectLayerByAttribute("micro_layer", "NEW_SELECTION", '"Sliver_Flag" = 1')

arcpy.management.Eliminate(
    in_features="micro_layer", 
    out_feature_class="microzones_cleaned_final",
    ex_features=taz_lines
)

ExecuteError: ERROR 000258: Output D:\ABM-INP-Microzones\Scripts\Scratch\Microzone_Cleanup\Microzone_Cleanup.gdb\microzones_cleaned_final already exists
Failed to execute (Eliminate).


Snap micro zones to the taz lines to get rid of micro cross over

In [ ]:
input_fc = "microzones_cleaned_final"
snapped_fc = "microzones_cleaned_final_snapped"
taz_lines = "taz_borders"


In [60]:
arcpy.management.CopyFeatures(input_fc, snapped_fc)
arcpy.management.Integrate(
    in_features=[[snapped_fc, 1]], 
    cluster_tolerance="1 Meters" # Adjust to "3 Feet" if your projection is in feet
)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzone_Cleanup\\Microzone_Cleanup.gdb\\microzones_cleaned_final_snapped'>

In [62]:
layer_name = "elim_layer"
arcpy.management.MakeFeatureLayer(snapped_fc, layer_name)
arcpy.management.SelectLayerByAttribute(layer_name, "NEW_SELECTION", '"Sliver_Flag" = 1')

arcpy.management.Eliminate(
        in_features=layer_name,
        out_feature_class="completed_microzones",
        ex_features=taz_lines
    )

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzone_Cleanup\\Microzone_Cleanup.gdb\\completed_microzones'>

Double check to see if there is any microzones that overlap a taz

In [63]:
microzones = "completed_microzones"
taz_polygons = "TAZ_Export"

layer_name = "final_layer"
arcpy.management.MakeFeatureLayer(microzones, layer_name)

arcpy.management.SelectLayerByLocation(
    in_layer=layer_name,
    overlap_type="CROSSED_BY_THE_OUTLINE_OF",
    select_features=taz_polygons
)

arcpy.management.CopyFeatures(layer_name, "crossing_errors")



<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzone_Cleanup\\Microzone_Cleanup.gdb\\crossing_errors'>